# Machine Learning Pipeline for Binary Target

The notebook is divided into small cells so each method can be selected, changed, or skipped independently.

**Before running:** Put `train.csv`, `test.csv`, and `submission_template.csv` in the notebook's working folder. Set `target_col`, `id_cols`, and `drop_cols` to match those files. Optional feature cells skip safely when their example columns are absent. The prediction pipeline reorders test features to match training features and imputes missing test columns.

**Model selection:** The sample compares AP and weighted F1, but the competition's exact metric should determine tuning and threshold selection. A held-out validation score is an estimate, not a guarantee of the private test score. RandomizedSearchCV is included; Optuna is optional and should only be added when its package is available. A probability blend of independently validated models can help, but keep it only if it improves out-of-fold or untouched validation performance.


## 1. Import Libraries

Core libraries for data preparation, modeling, validation, tuning, and interpretation.

**Purpose:** Load the tools used throughout the workflow so later cells can prepare data, train models, and evaluate predictions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, TimeSeriesSplit, GroupKFold, cross_val_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, RobustScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import average_precision_score, classification_report
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

## 2. Load Data

The default code assumes `train.csv` contains labelled rows and `test.csv` contains unlabelled rows. If the labelled data is split between `train.csv` and `test.csv`, while a separate `data_without_label.csv` is used for predictions, replace the two default file-loading lines with the commented `pd.concat` block below. Both files being combined must contain the target column.

**Purpose:** Load the standard labelled and unlabelled files, or uncomment the provided concatenation snippet when the labelled rows are split across two files.

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
submission = pd.read_csv("submission_template.csv")

# If the labelled rows are split across train.csv and test.csv, and the
# separate prediction file is data_without_label.csv, replace the two
# assignments above with this block:
# train = pd.concat(
#     [pd.read_csv("train.csv"), pd.read_csv("test.csv")],
#     ignore_index=True
# )
# test = pd.read_csv("data_without_label.csv")

In [ ]:
print("Train:", train.shape)
print("Test:", test.shape)
print("Submission:", submission.shape)

display(train.head())

## 3. Quick Data Check

**Purpose:** Inspect the dataset's structure and basic quality before making assumptions about columns or preprocessing.

In [ ]:
train.info()

In [ ]:
data_check = pd.DataFrame({
    "dtype": train.dtypes,
    "missing": train.isna().sum(),
    "missing_pct": (train.isna().mean() * 100).round(2),
    "unique": train.nunique(dropna=False)
})

display(data_check)

In [ ]:
print("Duplicate rows:", train.duplicated().sum())
display(train.describe(include="all").T)

## 4. Target Definition

Bad debt is defined as debt payment overdue for more than three months. Label 1 represents bad debt and label 0 represents no bad debt.

**Purpose:** Confirm what the positive and negative labels mean before selecting features or comparing models.

### Label Creation from Overdue Months

This transformation applies when the released data contains an overdue-duration field measured in months and no target label is supplied.

**Purpose:** Create a binary label from an overdue-duration field only when the released data does not already provide the target.

In [ ]:
# Optional: create a binary target only when the source provides overdue_months but no target.
# If your target already exists, leave it unchanged.
if "target" not in train.columns and "overdue_months" in train.columns:
    train["target"] = (train["overdue_months"] > 3).astype(int)
    print("Created target from overdue_months (> 3 months overdue).")
elif "target" in train.columns:
    print("Using the existing target column.")
else:
    print("No target created. Set target_col in the next section to the correct existing column.")


## 5. Select Target and Features

**Purpose:** Separate the target from input columns and exclude identifiers or fields that should not be used for prediction.

In [ ]:
# Change these names only if your files use different column names.
target_col = "target"  # labelled outcome column
id_cols = ["ID"]       # identifier columns retained for submission, excluded from training
drop_cols = []          # add any known leakage columns here

if target_col not in train.columns:
    raise KeyError(f"Target column {target_col!r} was not found in train.csv. Update target_col above.")


In [ ]:
print(train[target_col].value_counts())
print()
print((train[target_col].value_counts(normalize=True) * 100).round(2))

In [ ]:
# Build matching train and prediction feature tables. Extra columns in test are ignored;
# missing test features are created as missing values and imputed by the fitted pipeline.
feature_drop_cols = [c for c in [target_col] + id_cols + drop_cols if c in train.columns]
X = train.drop(columns=feature_drop_cols).copy()
y = train[target_col].copy()

test_feature_drop_cols = [c for c in id_cols + drop_cols + [target_col] if c in test.columns]
X_test = test.drop(columns=test_feature_drop_cols).copy()
missing_test_cols = [c for c in X.columns if c not in X_test.columns]
extra_test_cols = [c for c in X_test.columns if c not in X.columns]
if missing_test_cols:
    print("Test columns missing (will be imputed):", missing_test_cols)
if extra_test_cols:
    print("Test-only columns ignored:", extra_test_cols)
X_test = X_test.reindex(columns=X.columns)

if y.isna().any():
    raise ValueError("The training target contains missing values. Remove or resolve those rows before fitting.")
if set(pd.unique(y)) != {0, 1}:
    raise ValueError(f"Expected binary target values 0 and 1; found {sorted(pd.unique(y).tolist())}")

print(X.shape, y.shape, X_test.shape)


## 6. Select Numerical and Categorical Columns

**Purpose:** Group input columns by data type so the preprocessing steps can be applied appropriately.

In [ ]:
numerical_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

print("Numerical:", numerical_cols)
print("Categorical:", categorical_cols)

## 7. Data Quality Tests

These checks identify structural issues that can affect preprocessing, validation, and model stability.

**Purpose:** Look for unusual structure or values that may require a documented data-preparation decision.

### Missingness, Cardinality, and Constant Features

High missingness, constant variables, and very high-cardinality variables can reduce model quality or create unnecessary complexity.

**Purpose:** Find columns with extensive missing data, little variation, or many distinct values that may complicate modelling.

In [ ]:
feature_check = pd.DataFrame({
    "dtype": X.dtypes.astype(str),
    "missing_pct": (X.isna().mean() * 100).round(2),
    "unique": X.nunique(dropna=False),
    "unique_ratio": (X.nunique(dropna=False) / len(X)).round(4)
})

display(feature_check.sort_values("missing_pct", ascending=False))

In [ ]:
constant_features = feature_check[feature_check["unique"] <= 1]
high_cardinality = feature_check[feature_check["unique_ratio"] >= 0.80]

print("Constant features")
display(constant_features)

print("High-cardinality features")
display(high_cardinality)

### Infinite Values

Positive or negative infinity often appears after ratio or log transformations and should be handled before model fitting.

**Purpose:** Check for infinite numeric values that can cause preprocessing or model-fitting failures.

In [ ]:
numeric_data = X.select_dtypes(include="number")

infinite_count = pd.DataFrame({
    "positive_inf": np.isposinf(numeric_data).sum(),
    "negative_inf": np.isneginf(numeric_data).sum()
})

display(infinite_count[
    (infinite_count["positive_inf"] > 0) |
    (infinite_count["negative_inf"] > 0)
])

### Target Imbalance

A large difference between class frequencies can make accuracy misleading and may require class weighting, threshold adjustment, or resampling.

**Purpose:** Measure class proportions to understand whether the evaluation and training setup should account for unequal class counts.

In [ ]:
target_distribution = pd.DataFrame({
    "count": y.value_counts(),
    "percent": (y.value_counts(normalize=True) * 100).round(2)
})

display(target_distribution)

## 8. Outlier Tests

The IQR rule flags observations below Q1 - 1.5×IQR or above Q3 + 1.5×IQR. Outliers are diagnostic signals and are not automatically errors.

**Purpose:** Flag unusually extreme numeric observations for review; this diagnostic does not by itself justify deleting or changing them.

In [ ]:
outlier_rows = []

for col in numerical_cols:
    q1 = X[col].quantile(0.25)
    q3 = X[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_count = ((X[col] < lower) | (X[col] > upper)).sum()

    outlier_rows.append({
        "Feature": col,
        "Q1": q1,
        "Q3": q3,
        "Lower": lower,
        "Upper": upper,
        "Outliers": outlier_count,
        "Outlier_pct": outlier_count / len(X) * 100,
        "Skewness": X[col].skew()
    })

outlier_table = pd.DataFrame(outlier_rows)
display(outlier_table.sort_values("Outlier_pct", ascending=False).round(3))

## 9. Population Stability Index

PSI compares the distribution of a feature between train and test data. A common heuristic is PSI < 0.10 for small shift, 0.10–0.25 for moderate shift, and > 0.25 for substantial shift.

**Purpose:** Compare feature distributions in the labelled and unlabelled datasets to identify possible train-test shifts.

In [ ]:
def psi_numeric(train_series, test_series, bins=10):
    train_values = pd.to_numeric(train_series, errors="coerce").dropna()
    test_values = pd.to_numeric(test_series, errors="coerce").dropna()

    edges = np.unique(
        np.quantile(train_values, np.linspace(0, 1, bins + 1))
    )
    edges[0] = -np.inf
    edges[-1] = np.inf

    train_bin = pd.cut(train_values, bins=edges, include_lowest=True)
    test_bin = pd.cut(test_values, bins=edges, include_lowest=True)

    train_pct = train_bin.value_counts(
        normalize=True, sort=False
    )
    test_pct = test_bin.value_counts(
        normalize=True, sort=False
    ).reindex(train_pct.index, fill_value=0)

    train_pct = train_pct.clip(lower=1e-6)
    test_pct = test_pct.clip(lower=1e-6)

    psi = ((test_pct - train_pct) * np.log(test_pct / train_pct)).sum()
    return psi

In [ ]:
def psi_categorical(train_series, test_series):
    train_pct = (
        train_series.fillna("__MISSING__")
        .astype(str)
        .value_counts(normalize=True)
    )

    test_pct = (
        test_series.fillna("__MISSING__")
        .astype(str)
        .value_counts(normalize=True)
    )

    categories = train_pct.index.union(test_pct.index)

    train_pct = train_pct.reindex(categories, fill_value=0).clip(lower=1e-6)
    test_pct = test_pct.reindex(categories, fill_value=0).clip(lower=1e-6)

    psi = ((test_pct - train_pct) * np.log(test_pct / train_pct)).sum()
    return psi

In [ ]:
psi_rows = []

psi_numeric_cols = [
    col for col in numerical_cols
    if train[col].nunique(dropna=True) > 1
]

for col in psi_numeric_cols:
    psi_rows.append({
        "Feature": col,
        "Type": "Numerical",
        "PSI": psi_numeric(train[col], test[col])
    })

for col in categorical_cols:
    psi_rows.append({
        "Feature": col,
        "Type": "Categorical",
        "PSI": psi_categorical(train[col], test[col])
    })

psi_table = pd.DataFrame(psi_rows)
psi_table["Shift"] = pd.cut(
    psi_table["PSI"],
    bins=[-np.inf, 0.10, 0.25, np.inf],
    labels=["Small", "Moderate", "Substantial"]
)

display(psi_table.sort_values("PSI", ascending=False).round(4))

## 10. Feature Leakage Tests

Leakage occurs when a feature contains target information that would not be available at the real prediction time. Post-outcome fields such as overdue status, collections, write-offs, or recovery information are common leakage risks in credit problems.

**Purpose:** Review suspiciously predictive or post-outcome fields that might expose information unavailable at prediction time.

In [ ]:
leakage_words = [
    "target",
    "label",
    "bad_debt",
    "default",
    "overdue",
    "delinquent",
    "arrears",
    "collection",
    "writeoff",
    "write_off",
    "chargeoff",
    "charge_off",
    "recovery",
    "dpd"
]

name_leakage_candidates = [
    col for col in X.columns
    if any(word in col.lower() for word in leakage_words)
]

print("Suspicious feature names:")
print(name_leakage_candidates)

In [ ]:
numeric_with_target = train[numerical_cols + [target_col]].copy()

target_correlations = (
    numeric_with_target
    .corr(numeric_only=True)[target_col]
    .drop(target_col)
    .abs()
    .sort_values(ascending=False)
)

display(target_correlations.to_frame("abs_correlation_with_target").head(20))

In [ ]:
binary_candidates = [
    col for col in X.columns
    if train[col].nunique(dropna=False) <= 2
]

copy_scores = pd.Series({
    col: (
        train[col].astype(str) ==
        train[target_col].astype(str)
    ).mean()
    for col in binary_candidates
}).sort_values(ascending=False)

display(copy_scores.to_frame("target_match_rate").head(20))

## 11. Optional Feature Engineering

**Purpose:** Create additional inputs only when the released columns and business meaning make the transformation appropriate.

### Date Features

Convert a date column into components such as year, month, quarter, and day of week.

**Purpose:** Derive calendar components from a valid date field when those components could represent useful timing patterns.

In [ ]:
# Optional date features. This cell safely skips if the configured column is absent.
date_col = "application_date"  # replace with your actual date column
if date_col in train.columns and date_col in test.columns:
    train[date_col] = pd.to_datetime(train[date_col], errors="coerce")
    test[date_col] = pd.to_datetime(test[date_col], errors="coerce")
    for frame in (train, test):
        frame["year"] = frame[date_col].dt.year
        frame["month"] = frame[date_col].dt.month
        frame["quarter"] = frame[date_col].dt.quarter
        frame["day_of_week"] = frame[date_col].dt.dayofweek
    train = train.drop(columns=[date_col])
    test = test.drop(columns=[date_col])
else:
    print(f"Skipped date features: {date_col!r} is not present in both files.")


### Log Transformation

Useful for highly right-skewed positive numerical variables.

**Purpose:** Reduce the scale and skew of a suitable non-negative numeric variable while retaining its row-level information.

In [ ]:
# Optional log feature. Use only for a non-negative numeric variable.
col = "income"  # replace with a suitable numeric column
if col in train.columns and col in test.columns:
    train_values = pd.to_numeric(train[col], errors="coerce")
    test_values = pd.to_numeric(test[col], errors="coerce")
    if (train_values.dropna().ge(0).all() and test_values.dropna().ge(0).all()):
        train[col + "_log"] = np.log1p(train_values)
        test[col + "_log"] = np.log1p(test_values)
    else:
        print(f"Skipped log feature: {col!r} contains negative values.")
else:
    print(f"Skipped log feature: {col!r} is not present in both files.")


### Ratio Feature

Create a business ratio from two existing variables.

**Purpose:** Represent a meaningful relative quantity from two variables, taking care with zero or missing denominators.

In [ ]:
# Optional domain ratio. Replace both names with real numeric columns to enable it.
numerator_col = "numerator_col"
denominator_col = "denominator_col"
if all(c in train.columns and c in test.columns for c in [numerator_col, denominator_col]):
    for frame in (train, test):
        denominator = pd.to_numeric(frame[denominator_col], errors="coerce").replace(0, np.nan)
        frame["new_ratio"] = pd.to_numeric(frame[numerator_col], errors="coerce") / denominator
        frame["new_ratio"] = frame["new_ratio"].replace([np.inf, -np.inf], np.nan)
else:
    print("Skipped ratio feature: replace the placeholder column names with real columns to enable it.")


### Interaction Feature

Represent the combined effect of two numerical variables.

**Purpose:** Represent a possible combined effect of two numeric variables for models that can use the resulting feature.

In [ ]:
# Optional interaction. Replace the names with real numeric columns to enable it.
col_a = "col_a"
col_b = "col_b"
if all(c in train.columns and c in test.columns for c in [col_a, col_b]):
    train["interaction_feature"] = train[col_a] * train[col_b]
    test["interaction_feature"] = test[col_a] * test[col_b]
else:
    print("Skipped interaction feature: replace the placeholder column names with real columns to enable it.")


### Binning

Convert a continuous variable into interpretable groups.

**Purpose:** Group a continuous value into defined ranges when grouped levels are useful and the cut points are defensible.

In [ ]:
# Optional age band. Skips cleanly when age is not part of the supplied data.
bins = [0, 25, 40, 60, np.inf]
labels = ["<=25", "26-40", "41-60", "60+"]
if "age" in train.columns and "age" in test.columns:
    train["age_group"] = pd.cut(train["age"], bins=bins, labels=labels, include_lowest=True)
    test["age_group"] = pd.cut(test["age"], bins=bins, labels=labels, include_lowest=True)
else:
    print("Skipped age bands: 'age' is not present in both files.")


In [ ]:
# Refresh feature tables after the optional feature-engineering cells above.
feature_drop_cols = [c for c in [target_col] + id_cols + drop_cols if c in train.columns]
X = train.drop(columns=feature_drop_cols).copy()
y = train[target_col].copy()
test_feature_drop_cols = [c for c in id_cols + drop_cols + [target_col] if c in test.columns]
X_test = test.drop(columns=test_feature_drop_cols).copy().reindex(columns=X.columns)
numerical_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()
print("Final model inputs:", X.shape, "Prediction inputs:", X_test.shape)
print("Numerical features:", len(numerical_cols), "Categorical features:", len(categorical_cols))

## 12. Train and Validation Split

The code below is the default stratified random split. Choose one optional split below when observations repeat over time, customers have multiple rows, or both conditions apply.

**Purpose:** Reserve labelled observations for an honest model comparison while preserving class proportions where appropriate.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print(X_train.shape, X_valid.shape)
print(y_train.value_counts(normalize=True).round(3))
print(y_valid.value_counts(normalize=True).round(3))

### Optional TimeSeriesSplit

Use this instead of the random split above when rows represent observations over time and the goal is to evaluate later payment periods. The helper splits unique timestamps, keeping all rows from the same timestamp together. Replace `payment_date` with the observation/payment date column. For irregularly spaced dates, consider grouping into consistent periods such as weeks or months. If the same customer can appear in multiple periods, this evaluates future periods for a continuing portfolio; it does not hold out unseen customers.

**Purpose:** Use the latest chronological fold as validation when the goal is to estimate performance on later payment periods.

In [ ]:
def make_time_splits(X_part, time_values, n_splits=5, gap=0):
    """Return forward-only folds, keeping identical timestamps in one fold."""
    times = pd.Series(
        pd.to_datetime(time_values, errors="coerce"),
        index=X_part.index
    )
    if times.isna().any():
        raise ValueError("The selected time column contains missing or invalid dates.")

    ordered_times = pd.Index(times.unique()).sort_values()
    splitter = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    splits = []

    for train_time_idx, valid_time_idx in splitter.split(ordered_times):
        train_times = ordered_times[train_time_idx]
        valid_times = ordered_times[valid_time_idx]
        train_rows = np.flatnonzero(times.isin(train_times).to_numpy())
        valid_rows = np.flatnonzero(times.isin(valid_times).to_numpy())
        splits.append((train_rows, valid_rows))

    return splits


def make_time_based_holdout(X_all, y_all, time_values, n_splits=5, gap=0):
    """Use the final TimeSeriesSplit fold as a chronological holdout."""
    train_rows, valid_rows = make_time_splits(
        X_all, time_values, n_splits=n_splits, gap=gap
    )[-1]
    return (
        X_all.iloc[train_rows],
        X_all.iloc[valid_rows],
        y_all.iloc[train_rows],
        y_all.iloc[valid_rows],
    )


# To use time-based holdout instead of the random split above, uncomment:
# time_col = "payment_date"  # replace with the actual observation date column
# time_values = train.loc[X.index, time_col]
# X_train, X_valid, y_train, y_valid = make_time_based_holdout(
#     X, y, time_values, n_splits=5, gap=0
# )
# print("Train dates:", time_values.loc[X_train.index].min(), "to", time_values.loc[X_train.index].max())
# print("Validation dates:", time_values.loc[X_valid.index].min(), "to", time_values.loc[X_valid.index].max())
# print("Train positive rate:", y_train.mean())
# print("Validation positive rate:", y_valid.mean())

### Optional GroupKFold

Use GroupKFold instead of the random split when each customer has multiple rows and the intended use is to score customers who were not in training. Every row for one customer stays in the same fold. It does not enforce chronology, so use the time-based option as well if validation must represent a later period. Group folds are not stratified; inspect positive counts in every fold before interpreting metrics.

**Purpose:** Keep every observation from the same customer in one fold when the goal is to evaluate performance on unseen customers.

In [ ]:
def make_group_splits(X_part, y_part, group_values, n_splits=5):
    """Return folds that keep all rows from each group together."""
    groups = pd.Series(group_values, index=X_part.index)
    if groups.isna().any():
        raise ValueError("The selected group/customer column contains missing values.")
    if groups.nunique() < n_splits:
        raise ValueError("There must be at least n_splits distinct groups.")
    return list(GroupKFold(n_splits=n_splits).split(X_part, y_part, groups=groups))


def make_group_holdout(X_all, y_all, group_values, n_splits=5):
    """Use the last GroupKFold fold as a holdout of unseen groups."""
    train_rows, valid_rows = make_group_splits(
        X_all, y_all, group_values, n_splits=n_splits
    )[-1]
    return (X_all.iloc[train_rows], X_all.iloc[valid_rows],
            y_all.iloc[train_rows], y_all.iloc[valid_rows])


# To hold out customers instead of using the random split above, uncomment:
# group_col = "customer_id"  # replace with the actual customer identifier
# group_values = train.loc[X.index, group_col]
# X_train, X_valid, y_train, y_valid = make_group_holdout(
#     X, y, group_values, n_splits=5
# )
# print("Train/validation customers:", group_values.loc[X_train.index].nunique(), group_values.loc[X_valid.index].nunique())
# print("Train/validation positive rates:", y_train.mean(), y_valid.mean())

### Optional Time and Customer Group Split

Use this stricter option only when validation must represent a later period and customers in validation must be completely unseen during training. It takes the final chronological window, holds out a sample of customers from that window, and excludes all their earlier rows from training. This can discard many observations and produce a small or imbalanced validation set. Check row counts, date ranges, distinct customers, and positive counts. Uncomment this call instead of the random split only when it matches the deployment question.

**Purpose:** Combine chronological validation with customer holdout when the goal is later-period performance on customers not seen during training.

In [ ]:
def make_time_group_holdout(
    X_all, y_all, time_values, group_values,
    n_splits=5, gap=0, group_fraction=0.25, random_state=42
):
    """Hold out future rows from customers excluded entirely from training."""
    times = pd.Series(pd.to_datetime(time_values, errors="coerce"), index=X_all.index)
    groups = pd.Series(group_values, index=X_all.index)
    if times.isna().any() or groups.isna().any():
        raise ValueError("Time and group values must be present and valid for every row.")
    if not 0 < group_fraction < 1:
        raise ValueError("group_fraction must be between 0 and 1.")

    train_rows, valid_rows = make_time_splits(
        X_all, times, n_splits=n_splits, gap=gap
    )[-1]
    valid_candidate_idx = X_all.index[valid_rows]
    candidate_groups = groups.loc[valid_candidate_idx].drop_duplicates().to_numpy()
    if len(candidate_groups) < 2:
        raise ValueError("The final time window must contain at least two groups.")
    n_held = min(len(candidate_groups) - 1,
                 max(1, int(np.ceil(len(candidate_groups) * group_fraction))))
    held_groups = set(np.random.default_rng(random_state).choice(
        candidate_groups, size=n_held, replace=False
    ))

    valid_mask = times.isin(times.loc[valid_candidate_idx]) & groups.isin(held_groups)
    earlier_idx = X_all.index[train_rows]
    train_mask = times.isin(times.loc[earlier_idx]) & ~groups.isin(held_groups)
    train_idx = X_all.index[train_mask.to_numpy()]
    valid_idx = X_all.index[valid_mask.to_numpy()]
    if len(train_idx) == 0 or len(valid_idx) == 0:
        raise ValueError("The combined time/group split produced an empty partition.")
    return (X_all.loc[train_idx], X_all.loc[valid_idx],
            y_all.loc[train_idx], y_all.loc[valid_idx])


# For future periods and unseen customers, uncomment and adapt:
# time_col = "payment_date"
# group_col = "customer_id"
# X_train, X_valid, y_train, y_valid = make_time_group_holdout(
#     X, y, train.loc[X.index, time_col], train.loc[X.index, group_col],
#     n_splits=5, gap=0, group_fraction=0.25, random_state=42
# )
# print("Validation dates:", train.loc[X_valid.index, time_col].min(), "to", train.loc[X_valid.index, time_col].max())
# print("Train/validation customers:", train.loc[X_train.index, group_col].nunique(), train.loc[X_valid.index, group_col].nunique())
# print("Train/validation rows:", len(X_train), len(X_valid))
# print("Train/validation positive rates:", y_train.mean(), y_valid.mean())

## 13. Missing Value Methods

**Purpose:** Review alternative imputation rules and choose them within the fitted preprocessing pipeline.

### Median Imputation

Median is less sensitive to extreme values than mean.

**Purpose:** Fill numeric gaps with a robust centre that is less affected by extreme observations.

In [ ]:
numeric_median = SimpleImputer(strategy="median")

### Mean Imputation

Mean is simple and works best when extreme values are limited.

**Purpose:** Fill numeric gaps with the arithmetic average when the distribution and outlier profile make it suitable.

In [ ]:
numeric_mean = SimpleImputer(strategy="mean")

### Most Frequent Imputation

**Purpose:** Fill categorical gaps with the most common observed category.

In [ ]:
categorical_mode = SimpleImputer(strategy="most_frequent")

### Constant Category

**Purpose:** Represent missing categorical values as an explicit level when missingness itself may carry information.

In [ ]:
categorical_unknown = SimpleImputer(
    strategy="constant",
    fill_value="Unknown"
)

## 14. Encoding Methods

**Purpose:** Convert categories into numeric representations that the selected estimator can process.

### One-Hot Encoding

Create one binary column for each category.

**Purpose:** Create indicator columns for categories without assigning them an artificial order.

In [ ]:
onehot = OneHotEncoder(
    handle_unknown="ignore"
)

### Ordinal Encoding

Convert categories to integer codes. The numeric order does not automatically imply business order.

**Purpose:** Map categories to integer codes; use only when the representation is appropriate because the codes imply an order to many models.

In [ ]:
ordinal = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

## 15. Scaling Methods

**Purpose:** Review transformations that put numeric features on comparable scales for scale-sensitive models.

### StandardScaler

Center variables around zero and scale them by standard deviation.

**Purpose:** Centre numeric features and scale them by their standard deviation.

In [ ]:
standard_scaler = StandardScaler()

### RobustScaler

Scale using median and interquartile range, reducing the effect of extreme values.

**Purpose:** Scale numeric features using the median and interquartile range to reduce sensitivity to extreme values.

In [ ]:
robust_scaler = RobustScaler()

## 16. Preprocessing Pipelines

**Purpose:** Combine transformations so each model receives consistently prepared inputs and transformations are fitted on training data only.

### Standard Scaling + One-Hot Encoding

**Purpose:** Use imputation, numeric scaling, and categorical indicators together for models that benefit from scaled inputs.

In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", ))
])

preprocess_standard = ColumnTransformer([
    ("num", numeric_pipe, numerical_cols),
    ("cat", categorical_pipe, categorical_cols)
])

### No Scaling + One-Hot Encoding

Suitable for tree-based models that do not require feature scaling.

**Purpose:** Impute missing values and encode categories without scaling numeric values, a common setup for tree models.

In [ ]:
numeric_pipe_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", ))
])

preprocess_tree = ColumnTransformer([
    ("num", numeric_pipe_tree, numerical_cols),
    ("cat", categorical_pipe_tree, categorical_cols)
])

### No Scaling + Ordinal Encoding

A compact alternative when one-hot encoding creates too many columns.

**Purpose:** Use compact category codes when their imposed numeric structure is acceptable to the chosen model.

In [ ]:
numeric_pipe_ordinal = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe_ordinal = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

preprocess_ordinal = ColumnTransformer([
    ("num", numeric_pipe_ordinal, numerical_cols),
    ("cat", categorical_pipe_ordinal, categorical_cols)
])

## 17. Model Evaluation Template

The assessment focuses on F1 measures and predictive ranking. Average Precision is calculated from predicted probabilities.

**Purpose:** Create a consistent place to record validation metrics so candidate models can be compared on the same basis.

In [ ]:
results = []

## 18. Dummy Classifier

A simple benchmark showing how much the real models improve over a naive prediction strategy.

**Purpose:** Establish a simple baseline that shows how much value the feature-based models add over a naive prediction rule.

In [ ]:
dummy = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", DummyClassifier(strategy="prior"))
])

dummy.fit(X_train, y_train)

In [ ]:
pred_dummy = dummy.predict(X_valid)
prob_dummy = dummy.predict_proba(X_valid)[:, 1]

results.append({
    "Model": "Dummy",
    "F1_0": f1_score(y_valid, pred_dummy, pos_label=0),
    "F1_1": f1_score(y_valid, pred_dummy, pos_label=1),
    "Weighted_F1": f1_score(y_valid, pred_dummy, average="weighted"),
    "AP": average_precision_score(y_valid, prob_dummy)
})

display(pd.DataFrame(results).tail(1).round(4))

## 19. Logistic Regression

A linear probabilistic classifier. It is fast, interpretable, and benefits from scaled numerical variables.

**Purpose:** Fit a linear probabilistic classifier as an interpretable benchmark for the prepared features.

In [ ]:
logistic = Pipeline([
    ("preprocess", preprocess_standard),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

logistic.fit(X_train, y_train)

In [ ]:
pred_logistic = logistic.predict(X_valid)
prob_logistic = logistic.predict_proba(X_valid)[:, 1]

results.append({
    "Model": "Logistic Regression",
    "F1_0": f1_score(y_valid, pred_logistic, pos_label=0),
    "F1_1": f1_score(y_valid, pred_logistic, pos_label=1),
    "Weighted_F1": f1_score(y_valid, pred_logistic, average="weighted"),
    "AP": average_precision_score(y_valid, prob_logistic)
})

display(pd.DataFrame(results).tail(1).round(4))

## 20. Logistic Regression with Class Weight

Class weighting gives more importance to the minority class without creating new observations.

**Purpose:** Compare a cost-sensitive linear model that gives additional training weight to the less frequent class.

In [ ]:
logistic_balanced = Pipeline([
    ("preprocess", preprocess_standard),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

logistic_balanced.fit(X_train, y_train)

In [ ]:
pred_logistic_balanced = logistic_balanced.predict(X_valid)
prob_logistic_balanced = logistic_balanced.predict_proba(X_valid)[:, 1]

results.append({
    "Model": "Logistic Balanced",
    "F1_0": f1_score(y_valid, pred_logistic_balanced, pos_label=0),
    "F1_1": f1_score(y_valid, pred_logistic_balanced, pos_label=1),
    "Weighted_F1": f1_score(y_valid, pred_logistic_balanced, average="weighted"),
    "AP": average_precision_score(y_valid, prob_logistic_balanced)
})

display(pd.DataFrame(results).tail(1).round(4))

## 21. Decision Tree

A rule-based model that captures nonlinear patterns but can overfit.

**Purpose:** Fit a single tree to capture nonlinear splits and inspect the trade-off between flexibility and overfitting.

In [ ]:
decision_tree = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ))
])

decision_tree.fit(X_train, y_train)

In [ ]:
pred_tree = decision_tree.predict(X_valid)
prob_tree = decision_tree.predict_proba(X_valid)[:, 1]

results.append({
    "Model": "Decision Tree",
    "F1_0": f1_score(y_valid, pred_tree, pos_label=0),
    "F1_1": f1_score(y_valid, pred_tree, pos_label=1),
    "Weighted_F1": f1_score(y_valid, pred_tree, average="weighted"),
    "AP": average_precision_score(y_valid, prob_tree)
})

display(pd.DataFrame(results).tail(1).round(4))

## 22. Random Forest

An ensemble of decision trees that captures nonlinear relationships and usually reduces overfitting compared with a single tree.

**Purpose:** Fit an ensemble of randomized trees whose averaged predictions can capture nonlinear patterns.

In [ ]:
random_forest = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", RandomForestClassifier(
        n_estimators=300,
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

random_forest.fit(X_train, y_train)

In [ ]:
pred_rf = random_forest.predict(X_valid)
prob_rf = random_forest.predict_proba(X_valid)[:, 1]

results.append({
    "Model": "Random Forest",
    "F1_0": f1_score(y_valid, pred_rf, pos_label=0),
    "F1_1": f1_score(y_valid, pred_rf, pos_label=1),
    "Weighted_F1": f1_score(y_valid, pred_rf, average="weighted"),
    "AP": average_precision_score(y_valid, prob_rf)
})

display(pd.DataFrame(results).tail(1).round(4))

## 23. Random Forest with Class Weight

A simple imbalance treatment for tree ensembles.

**Purpose:** Compare a weighted forest to see whether class weighting changes minority-class performance.

In [ ]:
random_forest_balanced = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

random_forest_balanced.fit(X_train, y_train)

In [ ]:
pred_rf_balanced = random_forest_balanced.predict(X_valid)
prob_rf_balanced = random_forest_balanced.predict_proba(X_valid)[:, 1]

results.append({
    "Model": "Random Forest Balanced",
    "F1_0": f1_score(y_valid, pred_rf_balanced, pos_label=0),
    "F1_1": f1_score(y_valid, pred_rf_balanced, pos_label=1),
    "Weighted_F1": f1_score(y_valid, pred_rf_balanced, average="weighted"),
    "AP": average_precision_score(y_valid, prob_rf_balanced)
})

display(pd.DataFrame(results).tail(1).round(4))

## 24. Extra Trees

An ensemble similar to Random Forest but with more randomized split selection.

**Purpose:** Fit a highly randomized tree ensemble as another nonlinear candidate for comparison.

In [ ]:
extra_trees = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", ExtraTreesClassifier(
        n_estimators=300,
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

extra_trees.fit(X_train, y_train)

In [ ]:
pred_extra = extra_trees.predict(X_valid)
prob_extra = extra_trees.predict_proba(X_valid)[:, 1]

results.append({
    "Model": "Extra Trees",
    "F1_0": f1_score(y_valid, pred_extra, pos_label=0),
    "F1_1": f1_score(y_valid, pred_extra, pos_label=1),
    "Weighted_F1": f1_score(y_valid, pred_extra, average="weighted"),
    "AP": average_precision_score(y_valid, prob_extra)
})

display(pd.DataFrame(results).tail(1).round(4))

## 25. HistGradientBoosting

A boosting model that builds trees sequentially and focuses on correcting previous errors.

**Purpose:** Fit trees sequentially so later trees can improve on errors made by earlier trees.

In [ ]:
hist_gb = Pipeline([
    ("preprocess", preprocess_ordinal),
    ("model", HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=250,
        random_state=RANDOM_STATE
    ))
])

hist_gb.fit(X_train, y_train)

In [ ]:
pred_hist = hist_gb.predict(X_valid)
prob_hist = hist_gb.predict_proba(X_valid)[:, 1]

results.append({
    "Model": "HistGradientBoosting",
    "F1_0": f1_score(y_valid, pred_hist, pos_label=0),
    "F1_1": f1_score(y_valid, pred_hist, pos_label=1),
    "Weighted_F1": f1_score(y_valid, pred_hist, average="weighted"),
    "AP": average_precision_score(y_valid, prob_hist)
})

display(pd.DataFrame(results).tail(1).round(4))

## 26. XGBoost

A gradient boosting tree model with strong regularization and flexible hyperparameters. The package is optional; its model cells skip cleanly if it is not installed.

**Purpose:** Fit an optional gradient-boosted tree model when the package is available in the working environment.

In [ ]:
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False
    print("XGBoost is not installed; XGBoost sections will be skipped. Install it with: %pip install xgboost")


In [ ]:
if XGBOOST_AVAILABLE:
    xgb = Pipeline([
        ("preprocess", preprocess_tree),
        ("model", XGBClassifier(
            n_estimators=400,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ])

    xgb.fit(X_train, y_train)
else:
    print("Skipped: install xgboost to run this optional section.")


In [ ]:
if XGBOOST_AVAILABLE:
    pred_xgb = xgb.predict(X_valid)
    prob_xgb = xgb.predict_proba(X_valid)[:, 1]

    results.append({
        "Model": "XGBoost",
        "F1_0": f1_score(y_valid, pred_xgb, pos_label=0),
        "F1_1": f1_score(y_valid, pred_xgb, pos_label=1),
        "Weighted_F1": f1_score(y_valid, pred_xgb, average="weighted"),
        "AP": average_precision_score(y_valid, prob_xgb)
    })

    display(pd.DataFrame(results).tail(1).round(4))
else:
    print("Skipped: install xgboost to run this optional section.")


## 27. Compare Models

**Purpose:** Rank completed validation runs using the chosen metrics and use the comparison to support model selection.

### Optional probability blend

Averaging probabilities from different model families can improve ranking when their errors differ. This cell compares a simple RF/XGBoost blend on the same validation rows. Keep it only when it improves the competition metric on untouched validation or out-of-fold predictions; it does not change `final_model` automatically.

In [ ]:
if "prob_xgb" in globals():
    blend_probability = 0.5 * prob_rf + 0.5 * prob_xgb
    blend_prediction = (blend_probability >= 0.50).astype(int)
    print("RF + XGBoost probability blend")
    print("AP:", round(average_precision_score(y_valid, blend_probability), 4))
    print("Weighted F1:", round(f1_score(y_valid, blend_prediction, average="weighted"), 4))
else:
    print("Blend not evaluated because XGBoost validation probabilities are unavailable.")

In [ ]:
model_results = pd.DataFrame(results)
model_results = model_results.sort_values(
    ["AP", "Weighted_F1"],
    ascending=False
).reset_index(drop=True)

display(model_results.round(4))

## 28. Detailed Result for One Model

**Purpose:** Inspect class-level metrics and plots for the candidate selected for closer validation analysis.

In [ ]:
selected_model = random_forest
selected_pred = pred_rf
selected_prob = prob_rf

In [ ]:
print(classification_report(
    y_valid,
    selected_pred,
    digits=4,
    zero_division=0
))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_valid,
    selected_pred,
    values_format="d"
)

plt.title("Confusion Matrix")
plt.show()

In [ ]:
precision, recall, thresholds = precision_recall_curve(
    y_valid,
    selected_prob
)

plt.figure(figsize=(7, 5))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.grid(alpha=0.3)
plt.show()

## 29. Threshold Tuning

The probability threshold changes the trade-off between false negatives and false positives.

**Purpose:** Compare probability cutoffs to understand how the balance between positive-class precision and recall changes.

In [ ]:
threshold_rows = []

for threshold in np.arange(0.10, 0.91, 0.05):
    pred_threshold = (selected_prob >= threshold).astype(int)

    threshold_rows.append({
        "Threshold": round(threshold, 2),
        "Precision_1": precision_score(
            y_valid,
            pred_threshold,
            pos_label=1,
            zero_division=0
        ),
        "Recall_1": recall_score(
            y_valid,
            pred_threshold,
            pos_label=1,
            zero_division=0
        ),
        "F1_1": f1_score(
            y_valid,
            pred_threshold,
            pos_label=1,
            zero_division=0
        ),
        "Weighted_F1": f1_score(
            y_valid,
            pred_threshold,
            average="weighted",
            zero_division=0
        )
    })

threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table.round(4))

In [ ]:
# Set this to the exact competition metric before selecting a threshold.
# The default is weighted F1, which is reported throughout this notebook.
threshold_metric = "Weighted_F1"
if threshold_metric not in ["Weighted_F1", "F1_1"]:
    raise ValueError("threshold_metric must be 'Weighted_F1' or 'F1_1'.")
best_threshold_row = threshold_table.loc[threshold_table[threshold_metric].idxmax()]
display(best_threshold_row.to_frame().T.round(4))

## 30. Random Forest Tuning

Important Random Forest parameters include the number of trees, tree depth, minimum samples per split or leaf, feature sampling, and class weighting.

**Purpose:** Search a defined set of forest settings using cross-validation, then assess the selected configuration on validation data.

In [ ]:
rf_tune_model = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", RandomForestClassifier(
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

rf_param_dist = {
    "model__n_estimators": [200, 300, 400, 500, 700],
    "model__max_depth": [6, 8, 10, 12, 16, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 3, 5],
    "model__max_features": ["sqrt", "log2", None],
    "model__class_weight": [None, "balanced"]
}

In [ ]:
rf_search = RandomizedSearchCV(
    rf_tune_model,
    param_distributions=rf_param_dist,
    n_iter=5,
    scoring="average_precision",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)

print("Best CV AP:", rf_search.best_score_)
print("Best parameters:")
print(rf_search.best_params_)

In [ ]:
rf_best = rf_search.best_estimator_

pred_rf_best = rf_best.predict(X_valid)
prob_rf_best = rf_best.predict_proba(X_valid)[:, 1]

print("F1 0:", round(f1_score(y_valid, pred_rf_best, pos_label=0), 4))
print("F1 1:", round(f1_score(y_valid, pred_rf_best, pos_label=1), 4))
print("Weighted F1:", round(f1_score(y_valid, pred_rf_best, average="weighted"), 4))
print("AP:", round(average_precision_score(y_valid, prob_rf_best), 4))

## 31. XGBoost Tuning

Important XGBoost parameters include the number and depth of trees, learning rate, row and feature sampling, minimum child weight, and regularization.

**Purpose:** Search selected boosting settings using cross-validation when XGBoost is installed and runtime permits.

In [ ]:
if XGBOOST_AVAILABLE:
    negative_count = (y_train == 0).sum()
    positive_count = (y_train == 1).sum()
    class_ratio = negative_count / positive_count

    xgb_tune_model = Pipeline([
        ("preprocess", preprocess_tree),
        ("model", XGBClassifier(
            eval_metric="logloss",
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ])

    xgb_param_dist = {
        "model__n_estimators": [200, 300, 400, 500, 700],
        "model__max_depth": [3, 4, 5, 6, 8],
        "model__learning_rate": [0.01, 0.03, 0.05, 0.08, 0.10],
        "model__subsample": [0.7, 0.8, 0.9, 1.0],
        "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "model__min_child_weight": [1, 3, 5, 8],
        "model__reg_alpha": [0, 0.01, 0.1, 0.5],
        "model__reg_lambda": [1, 2, 5, 10],
        "model__scale_pos_weight": [1, class_ratio]
    }
else:
    print("Skipped: install xgboost to run this optional section.")


In [ ]:
if XGBOOST_AVAILABLE:
    xgb_search = RandomizedSearchCV(
        xgb_tune_model,
        param_distributions=xgb_param_dist,
        n_iter=5,
        scoring="average_precision",
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1
    )

    xgb_search.fit(X_train, y_train)

    print("Best CV AP:", xgb_search.best_score_)
    print("Best parameters:")
    print(xgb_search.best_params_)
else:
    print("Skipped: install xgboost to run this optional section.")


In [ ]:
if XGBOOST_AVAILABLE:
    xgb_best = xgb_search.best_estimator_

    pred_xgb_best = xgb_best.predict(X_valid)
    prob_xgb_best = xgb_best.predict_proba(X_valid)[:, 1]

    print("F1 0:", round(f1_score(y_valid, pred_xgb_best, pos_label=0), 4))
    print("F1 1:", round(f1_score(y_valid, pred_xgb_best, pos_label=1), 4))
    print("Weighted F1:", round(f1_score(y_valid, pred_xgb_best, average="weighted"), 4))
    print("AP:", round(average_precision_score(y_valid, prob_xgb_best), 4))
else:
    print("Skipped: install xgboost to run this optional section.")


## 32. Optuna Tuning for the RF + XGBoost Ensemble

This optional section searches Random Forest and XGBoost parameters with stratified cross-validation on `X_train` only. It uses Average Precision as the search objective by default; change `OPTUNA_SCORING` to match the competition metric. Each study can take several minutes or longer. If Optuna or XGBoost is unavailable, the notebook falls back to the existing model settings.

In [ ]:
try:
    import optuna
    OPTUNA_AVAILABLE = True
except Exception:
    OPTUNA_AVAILABLE = False
    print("Optuna is not installed. To enable tuning, run: %pip install optuna")

OPTUNA_TRIALS = 20  # increase for a longer search; reduce for a quicker first run
OPTUNA_FOLDS = 3
OPTUNA_SCORING = "average_precision"  # change to the exact competition metric when known
optuna_rf_study = None
optuna_xgb_study = None

In [ ]:
if OPTUNA_AVAILABLE:
    optuna_cv = StratifiedKFold(n_splits=OPTUNA_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    def rf_objective(trial):
        model = Pipeline([
            ("preprocess", preprocess_tree),
            ("model", RandomForestClassifier(
                n_estimators=trial.suggest_int("n_estimators", 250, 800, step=50),
                max_depth=trial.suggest_categorical("max_depth", [None, 6, 10, 14, 18, 24]),
                min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
                min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
                max_features=trial.suggest_float("max_features", 0.4, 1.0),
                class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),
                n_jobs=-1,
                random_state=RANDOM_STATE
            ))
        ])
        scores = cross_val_score(model, X_train, y_train, scoring=OPTUNA_SCORING, cv=optuna_cv, n_jobs=1)
        return float(scores.mean())

    optuna_rf_study = optuna.create_study(direction="maximize", study_name="random_forest")
    optuna_rf_study.optimize(rf_objective, n_trials=OPTUNA_TRIALS)
    print("Best RF CV score:", optuna_rf_study.best_value)
    print("Best RF parameters:", optuna_rf_study.best_params)

    if XGBOOST_AVAILABLE:
        positive_count = int((y_train == 1).sum())
        negative_count = int((y_train == 0).sum())
        class_ratio = negative_count / max(positive_count, 1)

        def xgb_objective(trial):
            model = Pipeline([
                ("preprocess", preprocess_tree),
                ("model", XGBClassifier(
                    n_estimators=trial.suggest_int("n_estimators", 250, 900, step=50),
                    max_depth=trial.suggest_int("max_depth", 3, 9),
                    learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
                    subsample=trial.suggest_float("subsample", 0.65, 1.0),
                    colsample_bytree=trial.suggest_float("colsample_bytree", 0.65, 1.0),
                    min_child_weight=trial.suggest_float("min_child_weight", 1, 12),
                    reg_alpha=trial.suggest_float("reg_alpha", 1e-5, 2.0, log=True),
                    reg_lambda=trial.suggest_float("reg_lambda", 0.5, 15.0, log=True),
                    scale_pos_weight=trial.suggest_float("scale_pos_weight", 1.0, max(1.0, class_ratio)),
                    eval_metric="logloss",
                    n_jobs=-1,
                    random_state=RANDOM_STATE
                ))
            ])
            scores = cross_val_score(model, X_train, y_train, scoring=OPTUNA_SCORING, cv=optuna_cv, n_jobs=1)
            return float(scores.mean())

        optuna_xgb_study = optuna.create_study(direction="maximize", study_name="xgboost")
        optuna_xgb_study.optimize(xgb_objective, n_trials=OPTUNA_TRIALS)
        print("Best XGBoost CV score:", optuna_xgb_study.best_value)
        print("Best XGBoost parameters:", optuna_xgb_study.best_params)
    else:
        print("XGBoost tuning skipped because XGBoost is unavailable.")
else:
    print("Optuna tuning skipped; the final models will use existing/default parameters.")

## 33. Cross-Validation

The code below uses stratified folds for the default random-split workflow. Optional time-aware folds for tuning and forward validation follow it.

**Purpose:** Estimate how consistently a pipeline performs across multiple stratified folds of the labelled data.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_scores = cross_val_score(
    random_forest,
    X,
    y,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1
)

print("Fold AP:", np.round(cv_scores, 4))
print("Mean AP:", round(cv_scores.mean(), 4))
print("Std AP:", round(cv_scores.std(), 4))

### Optional Time-Aware Cross-Validation for Tuning

If you use the time-based holdout, ordinary `cv=3` in the RandomizedSearchCV cells would still split training rows without respecting time. Build chronological folds from `X_train` only, then replace `cv=3` with `cv=time_tuning_splits` in the tuning cells. The example below is commented so the default random workflow remains unchanged.

**Purpose:** Keep hyperparameter search within earlier periods so the later validation window remains untouched.

In [ ]:
# For RandomizedSearchCV, create folds only inside the temporal training set:
# tuning_times = train.loc[X_train.index, time_col]
# time_tuning_splits = make_time_splits(
#     X_train, tuning_times, n_splits=3, gap=0
# )
# In the RandomizedSearchCV cells, replace cv=3 with:
# cv=time_tuning_splits

# For a separate forward-only cross-validation summary over all labelled rows:
# all_times = train.loc[X.index, time_col]
# all_time_splits = make_time_splits(X, all_times, n_splits=5, gap=0)
# time_cv_scores = cross_val_score(
#     random_forest, X, y, scoring="average_precision",
#     cv=all_time_splits, n_jobs=-1
# )
# print("Fold AP:", np.round(time_cv_scores, 4))
# print("Mean AP:", round(time_cv_scores.mean(), 4))

## 34. Tree Feature Importance

Tree importance indicates how strongly transformed features contributed to split decisions.

**Purpose:** Summarize which transformed features the fitted tree ensemble used for its split decisions.

In [ ]:
rf_preprocess = random_forest.named_steps["preprocess"]
rf_model = random_forest.named_steps["model"]

feature_names = rf_preprocess.get_feature_names_out()
feature_importance = rf_model.feature_importances_

importance_table = pd.DataFrame({
    "Feature": feature_names,
    "Importance": feature_importance
}).sort_values("Importance", ascending=False)

display(importance_table.head(20))

## 35. Logistic Regression Coefficients

Coefficients show the direction and relative magnitude of the linear relationship on the transformed feature scale.

**Purpose:** Inspect coefficient direction and relative magnitude on the transformed feature scale.

In [ ]:
log_preprocess = logistic.named_steps["preprocess"]
log_model = logistic.named_steps["model"]

feature_names_log = log_preprocess.get_feature_names_out()
coefficients = log_model.coef_[0]

coefficient_table = pd.DataFrame({
    "Feature": feature_names_log,
    "Coefficient": coefficients,
    "Abs_Coefficient": np.abs(coefficients)
}).sort_values("Abs_Coefficient", ascending=False)

display(coefficient_table.head(20))

## 36. Permutation Importance

Permutation importance measures how much validation performance decreases after shuffling one raw feature.

**Purpose:** Estimate how validation performance changes when one raw feature's values are shuffled.

In [ ]:
perm = permutation_importance(
    selected_model,
    X_valid,
    y_valid,
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

permutation_table = pd.DataFrame({
    "Feature": X_valid.columns,
    "Importance_Mean": perm.importances_mean,
    "Importance_Std": perm.importances_std
}).sort_values("Importance_Mean", ascending=False)

display(permutation_table.head(20))

## 37. Final Model

**Purpose:** Record the candidate pipeline and decision threshold that will be used for the final prediction workflow.

In [ ]:
# The final predictions average class-1 probabilities from the two fitted models when XGBoost is available.
USE_RF_XGB_ENSEMBLE = True
ENSEMBLE_WEIGHT_RF = 0.50  # 0.5 means equal probability weights; adjust only using validation evidence
final_model_name = "Random Forest + XGBoost probability ensemble" if XGBOOST_AVAILABLE and USE_RF_XGB_ENSEMBLE else "Random Forest"
if not 0.0 <= ENSEMBLE_WEIGHT_RF <= 1.0:
    raise ValueError("ENSEMBLE_WEIGHT_RF must be between 0 and 1.")
final_threshold = 0.50  # recalculated on the held-out validation set in the next section
print("Final prediction method:", final_model_name)

## 38. Refit on Full Training Data

Fit the selected pipeline again using all labelled observations.

**Purpose:** Refit the selected pipeline on all labelled observations after model selection is complete.

In [ ]:
# Prefer Optuna parameters; fall back to the earlier RandomizedSearchCV result, then notebook defaults.
rf_final_params = optuna_rf_study.best_params if optuna_rf_study is not None else (
    {key.replace("model__", ""): value for key, value in rf_search.best_params_.items()}
    if "rf_search" in globals() and hasattr(rf_search, "best_params_") else {}
)
rf_final_model = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", RandomForestClassifier(**rf_final_params, n_jobs=-1, random_state=RANDOM_STATE))
])

rf_final_model.fit(X_train, y_train)
rf_valid_probability = rf_final_model.predict_proba(X_valid)[:, 1]

if XGBOOST_AVAILABLE:
    xgb_final_params = optuna_xgb_study.best_params if optuna_xgb_study is not None else (
        {key.replace("model__", ""): value for key, value in xgb_search.best_params_.items()}
        if "xgb_search" in globals() and hasattr(xgb_search, "best_params_") else {
            "n_estimators": 400, "max_depth": 5, "learning_rate": 0.05,
            "subsample": 0.9, "colsample_bytree": 0.9
        }
    )
    xgb_final_model = Pipeline([
        ("preprocess", preprocess_tree),
        ("model", XGBClassifier(**xgb_final_params, eval_metric="logloss", n_jobs=-1, random_state=RANDOM_STATE))
    ])
    xgb_final_model.fit(X_train, y_train)
    xgb_valid_probability = xgb_final_model.predict_proba(X_valid)[:, 1]
else:
    xgb_final_model = None
    xgb_valid_probability = None

if USE_RF_XGB_ENSEMBLE and xgb_valid_probability is not None:
    validation_probability = ENSEMBLE_WEIGHT_RF * rf_valid_probability + (1 - ENSEMBLE_WEIGHT_RF) * xgb_valid_probability
else:
    validation_probability = rf_valid_probability

# Pick the threshold on held-out validation predictions using weighted F1.
threshold_candidates = np.arange(0.10, 0.91, 0.01)
threshold_scores = [
    f1_score(y_valid, (validation_probability >= threshold).astype(int), average="weighted")
    for threshold in threshold_candidates
]
final_threshold = float(threshold_candidates[int(np.argmax(threshold_scores))])
print("Validation weighted F1:", round(max(threshold_scores), 4))
print("Chosen threshold:", final_threshold)

# Refit both members on every labelled row before generating submission predictions.
rf_final_model.fit(X, y)
if xgb_final_model is not None:
    xgb_final_model.fit(X, y)
final_model = rf_final_model


## 39. Predict Test Data

Generate probabilities first, then convert them into binary labels using the selected threshold.

**Purpose:** Generate test-set predictions from the refitted pipeline using the chosen probability threshold.

In [ ]:
rf_test_probability = rf_final_model.predict_proba(X_test)[:, 1]
if USE_RF_XGB_ENSEMBLE and xgb_final_model is not None:
    xgb_test_probability = xgb_final_model.predict_proba(X_test)[:, 1]
    test_probability = ENSEMBLE_WEIGHT_RF * rf_test_probability + (1 - ENSEMBLE_WEIGHT_RF) * xgb_test_probability
else:
    test_probability = rf_test_probability

test_prediction = (test_probability >= final_threshold).astype(int)

In [ ]:
prediction_preview = pd.DataFrame({
    "Probability": test_probability[:10],
    "Prediction": test_prediction[:10]
})

display(prediction_preview)
print("Positive rate:", round(test_prediction.mean(), 4))

## 40. Create Submission

**Purpose:** Place predictions into the supplied submission structure and save the required output file.

In [ ]:
prediction_col = target_col
if len(submission) != len(test_prediction):
    raise ValueError(
        f"Submission template has {len(submission)} rows but test data has {len(test_prediction)} rows. "
        "Use the correct submission_template.csv."
    )
submission[prediction_col] = test_prediction

display(submission.head())
print(submission.shape)
print(submission[prediction_col].value_counts())


In [ ]:
submission.to_csv(
    "submission_final.csv",
    index=False
)

## 41. Result Tables for the Report

**Purpose:** Collect completed validation metrics and final model details for concise reporting.

In [ ]:
display(model_results.round(4))

In [ ]:
final_summary = pd.DataFrame({
    "Item": [
        "Final model",
        "Validation method",
        "Threshold",
        "Train rows",
        "Test rows",
        "Number of features"
    ],
    "Value": [
        final_model_name,
        "80/20 stratified split",
        final_threshold,
        len(train),
        len(test),
        X.shape[1]
    ]
})

display(final_summary)

## 42. SHAP Analysis

SHAP values explain how individual features contribute to model predictions. Global plots show important drivers and direction of influence; local plots explain one observation. SHAP supports model interpretation but does not establish causation.

**Purpose:** Use SHAP as an optional model-interpretation method; its explanations describe model behaviour, not causation.

### XGBoost SHAP

**Purpose:** Explain the fitted boosting model's global and selected individual predictions using SHAP values.

In [ ]:
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP is not installed; interpretation cells will be skipped. Install it with: %pip install shap")


In [ ]:
if SHAP_AVAILABLE and XGBOOST_AVAILABLE and "xgb_best" in globals():
    xgb_shap_pipeline = xgb_best
    xgb_shap_preprocess = xgb_shap_pipeline.named_steps["preprocess"]
    xgb_shap_model = xgb_shap_pipeline.named_steps["model"]

    X_shap_sample = X_valid.sample(
        n=min(500, len(X_valid)),
        random_state=RANDOM_STATE
    )

    X_shap_matrix = xgb_shap_preprocess.transform(X_shap_sample)
    xgb_feature_names = xgb_shap_preprocess.get_feature_names_out()

    X_shap = pd.DataFrame(
        X_shap_matrix,
        columns=xgb_feature_names,
        index=X_shap_sample.index
    )
else:
    print("Skipped: this optional plot requires SHAP, XGBoost, and the XGBoost tuning cells.")


In [ ]:
if SHAP_AVAILABLE and XGBOOST_AVAILABLE and "xgb_best" in globals():
    xgb_explainer = shap.TreeExplainer(xgb_shap_model)
    xgb_shap_values = xgb_explainer(X_shap)
else:
    print("Skipped: this optional plot requires SHAP, XGBoost, and the XGBoost tuning cells.")


In [ ]:
if SHAP_AVAILABLE and XGBOOST_AVAILABLE and "xgb_best" in globals():
    shap.plots.bar(
        xgb_shap_values,
        max_display=15
    )
else:
    print("Skipped: this optional plot requires SHAP, XGBoost, and the XGBoost tuning cells.")


In [ ]:
if SHAP_AVAILABLE and XGBOOST_AVAILABLE and "xgb_best" in globals():
    shap.plots.beeswarm(
        xgb_shap_values,
        max_display=15
    )
else:
    print("Skipped: this optional plot requires SHAP, XGBoost, and the XGBoost tuning cells.")


In [ ]:
if SHAP_AVAILABLE and XGBOOST_AVAILABLE and "xgb_best" in globals():
    shap.plots.waterfall(
        xgb_shap_values[0],
        max_display=15
    )
else:
    print("Skipped: optional SHAP model or dependency is unavailable.")


### Random Forest SHAP

**Purpose:** Explain the fitted forest's global and selected individual predictions using SHAP values.

In [ ]:
if SHAP_AVAILABLE and "rf_best" in globals():
    rf_shap_pipeline = rf_best
    rf_shap_preprocess = rf_shap_pipeline.named_steps["preprocess"]
    rf_shap_model = rf_shap_pipeline.named_steps["model"]

    X_shap_rf_matrix = rf_shap_preprocess.transform(X_shap_sample)
    rf_feature_names = rf_shap_preprocess.get_feature_names_out()

    X_shap_rf = pd.DataFrame(
        X_shap_rf_matrix,
        columns=rf_feature_names,
        index=X_shap_sample.index
    )
else:
    print("Skipped: this optional plot requires SHAP and the Random Forest tuning cells.")


In [ ]:
if SHAP_AVAILABLE and "rf_best" in globals():
    rf_explainer = shap.TreeExplainer(rf_shap_model)
    rf_shap_values = rf_explainer(X_shap_rf)

    rf_shap_class1 = rf_shap_values[:, :, 1]
else:
    print("Skipped: this optional plot requires SHAP and the Random Forest tuning cells.")


In [ ]:
if SHAP_AVAILABLE and "rf_best" in globals():
    shap.plots.bar(
        rf_shap_class1,
        max_display=15
    )
else:
    print("Skipped: this optional plot requires SHAP and the Random Forest tuning cells.")


In [ ]:
if SHAP_AVAILABLE and "rf_best" in globals():
    shap.plots.beeswarm(
        rf_shap_class1,
        max_display=15
    )
else:
    print("Skipped: this optional plot requires SHAP and the Random Forest tuning cells.")


In [ ]:
if SHAP_AVAILABLE and "rf_best" in globals():
    shap.plots.waterfall(
        rf_shap_class1[0],
        max_display=15
    )
else:
    print("Skipped: optional SHAP model or dependency is unavailable.")


## 43. Report Outline

### Best model architecture and workflow
- Data preparation
- Missing-value treatment
- Encoding
- Scaling
- Feature engineering
- Class imbalance treatment
- Final algorithm
- Optuna and RandomizedSearchCV tuning
- Probability blend and weights
- Main hyperparameters
- Validation strategy
- Leakage prevention
- Final refit

### Results analysis
- F1 for label 0
- F1 for label 1
- Weighted F1
- Average Precision
- Confusion matrix
- Precision-recall trade-off
- Model comparison
- False negative implication
- False positive implication
- Limitation
- Realistic improvement
- Feature importance

**Purpose:** Organize the final report around the selected workflow, validation evidence, interpretation, and limitations.